In [1]:
import os
from datetime import datetime
import sys
from pathlib import Path
import shutil
import zipfile
import rarfile
import tarfile
import py7zr
import pandas as pd

from thefuzz import process
import tqdm
from tqdm.auto import tqdm

In [2]:
def move_file_to_dir(file, dest_dir):
    file_name = file.name
    dest_path = dest_dir / file_name
    shutil.move(file, dest_path)

In [3]:
def dir_size(dir_path):
    size = 0
    for file in dir_path.rglob('*'):
        if file.is_file():
            size += file.stat().st_size
    return size

In [4]:
def manage_zip(zip_file, curr_dir, parent_dir, irrelevant_files_dir,
               unprocessed_arch_dir, processed_arch_dir, base_dir):
    #extract zip_file to 00_new_folder
    try:
        new_dir = parent_dir / zip_file.stem
        with zipfile.ZipFile(zip_file, 'r') as zf:
            zf.extractall(new_dir)
        move_file_to_dir(zip_file, processed_arch_dir)
        manage_files(curr_dir, new_dir, irrelevant_files_dir,
                     unprocessed_arch_dir, processed_arch_dir, base_dir)
        shutil.rmtree(new_dir)
    except Exception as e:
        move_file_to_dir(zip_file, unprocessed_arch_dir)

In [5]:
def manage_rar(rar_file, curr_dir, parent_dir, irrelevant_files_dir,
               unprocessed_arch_dir, processed_arch_dir, base_dir):
    try:
        new_dir = parent_dir / rar_file.stem
        with rarfile.RarFile(item) as rf:
            rf.extractall(new_dir)
        move_file_to_dir(rar_file, processed_arch_dir)
        manage_files(curr_dir, new_dir, irrelevant_files_dir,
                     unprocessed_arch_dir, processed_arch_dir, base_dir)
        shutil.rmtree(new_dir)
    except Exception as e:
        move_file_to_dir(rar_file, unprocessed_arch_dir)

In [6]:
def manage_tar(tar_file, curr_dir, parent_dir, irrelevant_files_dir,
               unprocessed_arch_dir, processed_arch_dir, base_dir):
    try:
        new_dir = parent_dir / tar_file.stem
        with tarfile.open(tar_file, 'r') as tar:
            tar.extractall(new_dir)
        move_file_to_dir(tar_file, processed_arch_dir)
        manage_files(curr_dir, new_dir, irrelevant_files_dir,
                     unprocessed_arch_dir, processed_arch_dir, base_dir)
        shutil.rmtree(new_dir)
    except Exception as e:
        move_file_to_dir(tar_file, unprocessed_arch_dir)

In [7]:
def manage_7z(seven_zip_file, curr_dir, parent_dir, irrelevant_files_dir,
               unprocessed_arch_dir, processed_arch_dir, base_dir):
    try:
        new_dir = parent_dir / seven_zip_file.stem
        with py7zr.SevenZipFile(seven_zip_file, mode='r') as seven_zip:
            seven_zip.extractall(path=new_dir)
        move_file_to_dir(seven_zip_file, processed_arch_dir)
        manage_files(curr_dir, new_dir, irrelevant_files_dir,
                     unprocessed_arch_dir, processed_arch_dir, base_dir)
        shutil.rmtree(new_dir)
    except Exception as e:
        move_file_to_dir(seven_zip_file, unprocessed_arch_dir)

In [8]:
def manage_csv(csv_file, curr_dir, parent_dir, irrelevant_files_dir,
               unprocessed_arch_dir, processed_arch_dir, base_dir):
    global counter
    csv_parent = csv_file.parent
    if "_" in csv_file.stem:
        # handle csv files with name prefix
        common_pattern = csv_file.stem.split("_")[0]
        csv_files = [f for f in csv_parent.glob("*.csv") if common_pattern in f.stem]
    else:
        # handle csv files without name prefix
        csv_files = [f for f in csv_parent.glob("*.csv")]
        
    counter += 1
    csv_dest = base_dir / f"mm_{str(counter)}"
    os.makedirs(csv_dest, exist_ok=True)
    
    for csv_file in csv_files:
        move_file_to_dir(csv_file, csv_dest)

In [9]:
def manage_files(curr_dir, parent_dir, irrelevant_files_dir, 
                 unprocessed_arch_dir, processed_arch_dir, base_dir):
    
    for item in tqdm(parent_dir.iterdir(), desc="Processing..."):
        if item.exists():
            if item.is_file():
                item_extn = item.suffix.lower().strip()
                if item_extn == ".zip":
                    manage_zip(item, curr_dir, parent_dir, irrelevant_files_dir, 
                                 unprocessed_arch_dir, processed_arch_dir, base_dir)
                elif item_extn == ".rar":
                    manage_rar(item, curr_dir, parent_dir, irrelevant_files_dir, 
                                 unprocessed_arch_dir, processed_arch_dir, base_dir)
                elif item_extn == ".tar":
                    manage_tar(item, curr_dir, parent_dir, irrelevant_files_dir, 
                                 unprocessed_arch_dir, processed_arch_dir, base_dir)
                elif item_extn == ".7z":
                    manage_7z(item, curr_dir, parent_dir, irrelevant_files_dir, 
                                 unprocessed_arch_dir, processed_arch_dir, base_dir)
                elif item_extn == ".csv":
                    manage_csv(item, curr_dir, parent_dir, irrelevant_files_dir, 
                                 unprocessed_arch_dir, processed_arch_dir, base_dir)
                else:
                    #all other irrelevant files
                    move_file_to_dir(item, irrelevant_files_dir)
            elif item.is_dir():
                # if item size > 0, then move all files from inside of item to item.parent
                item_size = dir_size(item)
                if item_size > 0:
                    manage_files(curr_dir, item, irrelevant_files_dir, unprocessed_arch_dir, processed_arch_dir, base_dir)

                # if item is empty or size=0, delete item
                item_is_empty = not any(item.iterdir())
                item_size = dir_size(item)
                if item_is_empty or item_size==0:
                    shutil.rmtree(item)

In [10]:
def main():
    curr_dir = Path.cwd()
    parent_dir = curr_dir / "01_parent_directory"
    irrelevant_files_dir = curr_dir / "02_irrelevant_files"
    unprocessed_arch_dir = curr_dir / "03_unprocessed_archives"
    processed_arch_dir = curr_dir / "04_processed_archives"
    base_dir = curr_dir / "05_base_folder"

    os.makedirs(irrelevant_files_dir, exist_ok=True)
    os.makedirs(unprocessed_arch_dir, exist_ok=True)
    os.makedirs(processed_arch_dir, exist_ok=True)
    os.makedirs(base_dir, exist_ok=True)
    
    manage_files(curr_dir, parent_dir, irrelevant_files_dir, unprocessed_arch_dir, processed_arch_dir, base_dir)

In [11]:
counter = 100000
main()

Processing...: 0it [00:00, ?it/s]